# GPT-1 구현 QUEST

원조 Transformer(Vaswani et al., 2017) 코드를 수정하여 **GPT-1**(Radford et al., 2018, *Improving Language Understanding by Generative Pre-Training*)을 구현한다.

본 과제 범위는 **사전학습(pre-training)** 까지로 한정한다. 즉 라벨 없는 텍스트로 **다음 토큰을 예측**하는 언어모델 학습만 다루며, 논문 3.2/3.3의 fine-tuning 및 task-specific 입력 변형은 다루지 않는다.

## 1. Transformer와 비교해 변경이 필요한 부분 (아키텍처 변경 서술)

원조 Transformer는 **Encoder–Decoder** 구조이며, 번역 같은 seq2seq 과제를 위해 설계되었다. GPT-1은 여기서 **Decoder 계열 블록만** 떼어내어 **언어모델(다음 단어 예측)** 로 재구성한다. 블록 단위 변경 사항은 다음과 같다.

| 구성 요소 | 원조 Transformer | GPT-1 (본 구현) |
|---|---|---|
| 전체 구조 | Encoder + Decoder | **Decoder 블록만** (Encoder 제거) |
| Cross-Attention | Decoder에 존재 (Encoder 출력을 참조) | **제거** — 참조할 Encoder가 없음 |
| Self-Attention | Encoder=양방향, Decoder=masked | **Masked(causal) self-attention만** → 단방향 |
| 블록 구성 | (Self-Attn → Cross-Attn → FFN) | **(Masked Self-Attn → FFN)** 2부품으로 축소 |
| 위치 정보 | Sinusoidal positional encoding (고정) | **학습되는 Position Embedding** |
| 학습 목표 | seq2seq (예: 번역) | **Causal LM** (앞 토큰들로 다음 토큰 예측) |
| 층 수 | 6층 | 논문 12층 (본 실습은 CPU 환경상 축소) |

**핵심 변경 요약**
1. Encoder 전체와 Decoder 내부의 **cross-attention 서브층을 제거**한다. 그 결과 블록은 *masked multi-head self-attention → feed-forward* 두 부품만 남는다.
2. Self-attention에 **causal mask**를 적용하여 각 위치가 **자기 자신과 과거 토큰만** 보도록 한다(미래 토큰 차단). 이것이 GPT를 단방향 생성 모델로 만드는 정체성이다.
3. 위치 정보는 sinusoidal 대신 **학습 가능한 position embedding**으로 더한다.
4. 출력은 어휘 전체에 대한 softmax 분포로, **다음 토큰을 예측**하도록 학습한다.

## 0. 환경 설정 및 하이퍼파라미터

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow:", tf.__version__)

I0000 00:00:1782188172.869210     599 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782188172.869721     599 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782188172.916212     599 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1782188174.105066     599 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782188174.105412     599 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow: 2.21.0


In [2]:
# ── 하이퍼파라미터 (논문 스펙을 CPU 실습 규모로 축소) ──
SEQ_LEN   = 64     # 한 번에 보는 토큰 길이 (context window)
D_MODEL   = 128    # 임베딩/은닉 차원 (논문 768)
N_HEADS   = 4      # 멀티헤드 수      (논문 12)
N_LAYERS  = 4      # Transformer 블록 층 수 (논문 12)
D_FF      = 512    # FFN 내부 차원    (논문 3072)
DROPOUT   = 0.1
BATCH     = 32
EPOCHS    = 8
print("hidden:", D_MODEL, "| heads:", N_HEADS, "| layers:", N_LAYERS)

hidden: 128 | heads: 4 | layers: 4


## 2. 입력 형태에 맞게 전처리 (Decoder 기반 생성모델용 데이터 변형)

GPT는 "앞 토큰들 → 다음 토큰"을 학습하는 생성모델이다. 따라서 텍스트를 토큰 시퀀스로 만든 뒤,
**입력 x = 토큰[0:n]**, **정답 y = 토큰[1:n+1]** 처럼 **한 칸 밀어서(shift)** 쌍을 만든다.
즉 매 위치에서 "바로 다음 토큰"이 정답이 된다.

토크나이저는 별도 다운로드 없이 재현 가능하도록 **문자 단위(char-level)** 로 구성한다.
(BPE 대신 char-level을 써도 위치정보·causal mask·다음토큰 생성이라는 GPT의 핵심 동작 검증에는 충분하다.)

In [3]:
# ── 내장 학습 텍스트 (외부 의존성 없이 재현 가능하도록) ──
text = (
    "the quick brown fox jumps over the lazy dog. "
    "a language model learns to predict the next token from previous tokens. "
    "generative pre-training teaches the model the structure of language. "
    "the transformer decoder uses masked self attention to look only at the past. "
    "deep learning models improve with more data and more parameters. "
) * 60   # 반복하여 학습량 확보

# 문자 단위 vocabulary 구성
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}   # char -> id
itos = {i: c for c, i in stoi.items()}       # id -> char
VOCAB = len(chars)
print("vocab size:", VOCAB)
print("total chars:", len(text))

def encode(s): return [stoi[c] for c in s]
def decode(ids): return "".join(itos[int(i)] for i in ids)

data = np.array(encode(text), dtype=np.int32)
print("encoded sample:", data[:20])

vocab size: 29
total chars: 19680
encoded sample: [22 10  7  0 19 23 11  5 13  0  4 20 17 25 16  0  8 17 26  0]


In [4]:
# ── (입력, 정답) 쌍 만들기: 한 칸 shift ──
# x = tokens[i : i+SEQ_LEN],  y = tokens[i+1 : i+SEQ_LEN+1]
def make_pairs(data, seq_len):
    xs, ys = [], []
    for i in range(0, len(data) - seq_len - 1, seq_len):
        xs.append(data[i : i + seq_len])
        ys.append(data[i + 1 : i + seq_len + 1])
    return np.array(xs), np.array(ys)

X, Y = make_pairs(data, SEQ_LEN)
print("X shape:", X.shape, "| Y shape:", Y.shape)
print("\n예시 (x를 한 칸 밀면 y가 됨):")
print("x:", decode(X[0][:30]))
print("y:", decode(Y[0][:30]))

X shape: (307, 64) | Y shape: (307, 64)

예시 (x를 한 칸 밀면 y가 됨):
x: the quick brown fox jumps over
y: he quick brown fox jumps over 


## 3. 입력 블록 — 위치 정보 추가 (GPT 논문 식 (2): h0 = U·We + Wp)

논문의 입력층은 **토큰 임베딩 We** 와 **위치 임베딩 Wp** 를 더해 첫 표현 h0를 만든다.

$$h_0 = U W_e + W_p$$

원조 Transformer의 고정 sinusoidal 인코딩과 달리, GPT는 **학습되는 position embedding**을 사용한다.
아래 `TokenAndPositionEmbedding` 층이 토큰 임베딩과 위치 임베딩을 각각 두고 **더해서** 반환한다.

In [5]:
class TokenAndPositionEmbedding(layers.Layer):
    """토큰 임베딩(We) + 학습되는 위치 임베딩(Wp). 논문 식 (2)의 h0 = U·We + Wp."""
    def __init__(self, seq_len, vocab_size, d_model, **kwargs):
        super().__init__(**kwargs)
        self.tok_emb = layers.Embedding(input_dim=vocab_size, output_dim=d_model, name="token_emb")    # We
        self.pos_emb = layers.Embedding(input_dim=seq_len,    output_dim=d_model, name="position_emb") # Wp (학습됨)
        self.seq_len = seq_len

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)  # 0,1,2,...,n-1
        return self.tok_emb(x) + self.pos_emb(positions)               # 토큰벡터 + 위치벡터

    def get_config(self):
        return {"seq_len": self.seq_len}

# 동작 확인
_emb = TokenAndPositionEmbedding(SEQ_LEN, VOCAB, D_MODEL)
_out = _emb(X[:2])
print("embedding output shape:", _out.shape, " -> (batch, seq_len, d_model)")

embedding output shape: (2, 64, 128)  -> (batch, seq_len, d_model)


E0000 00:00:1782188175.491377     599 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 4. GPT 모델 구성 — Transformer 코드를 GPT-1로 수정

### 4-1. Decoder 블록 (cross-attention 제거 + causal mask)

원조 Transformer 디코더 블록은 *(masked self-attn → **cross-attn** → FFN)* 이지만,
GPT는 Encoder가 없으므로 **cross-attention을 삭제**한다. 남는 구성은 **(masked self-attn → FFN)** 이다.
`MultiHeadAttention` 호출 시 `use_causal_mask=True`로 **미래 토큰을 차단**한다(causal mask).

In [6]:
class GPTBlock(layers.Layer):
    """GPT-1의 Transformer 블록: Masked Multi-Head Self-Attention -> FFN.
       (원조 Transformer 디코더에서 cross-attention 서브층을 제거한 형태)"""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn = layers.MultiHeadAttention(num_heads=n_heads, key_dim=d_model // n_heads)
        self.ffn = keras.Sequential([
            layers.Dense(d_ff, activation="gelu"),   # 논문은 GELU 사용
            layers.Dense(d_model),
        ])
        self.ln1 = layers.LayerNormalization(epsilon=1e-5)
        self.ln2 = layers.LayerNormalization(epsilon=1e-5)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        # 1) Masked self-attention (causal mask로 미래 차단). cross-attention 없음.
        attn_out = self.attn(x, x, use_causal_mask=True, training=training)
        x = self.ln1(x + self.drop1(attn_out, training=training))   # residual + norm
        # 2) Position-wise feed-forward
        ffn_out = self.ffn(x)
        x = self.ln2(x + self.drop2(ffn_out, training=training))    # residual + norm
        return x

### 4-2. 전체 GPT 모델 조립

입력 → (토큰+위치 임베딩) → GPT 블록 × N_LAYERS → 마지막 Dense(VOCAB)로 **다음 토큰 분포** 출력.
마지막 Dense가 논문 식 (2)의 `softmax(h_n · We^T)` 에 해당하는 출력층 역할을 한다
(여기서는 가중치 공유 대신 별도 Dense로 단순화).

In [7]:
def build_gpt():
    inputs = keras.Input(shape=(SEQ_LEN,), dtype="int32", name="token_ids")
    x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB, D_MODEL)(inputs)   # h0 = We + Wp
    for i in range(N_LAYERS):
        x = GPTBlock(D_MODEL, N_HEADS, D_FF, DROPOUT, name=f"gpt_block_{i+1}")(x)
    logits = layers.Dense(VOCAB, name="lm_head")(x)                 # 각 위치마다 다음 토큰 점수
    return keras.Model(inputs, logits, name="GPT1")

model = build_gpt()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)
# ── 평가기준 4: model.summary 캡처 ──
model.summary()

Model: "GPT1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)          │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_1  │ (None, 64, 128)        │        11,904 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_1 (GPTBlock)          │ (None, 64, 128)        │       198,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_2 (GPTBlock)          │ (None, 64, 128)        │       198,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_3 (GPTBlock)          │ (None, 64, 128)        │       198,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_4 (GPTBlock)          │ (None, 64, 128)        │       198,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lm_head (Dense)                 │ (None, 64, 29)         │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 808,733 (3.09 MB)

 Trainable params: 808,733 (3.09 MB)

 Non-trainable params: 0 (0.00 B)

### 4-3. 학습 (model.fit 캡처)

In [8]:
history = model.fit(X, Y, batch_size=BATCH, epochs=EPOCHS, validation_split=0.1, verbose=2)

Epoch 1/8


9/9 - 10s - 1s/step - loss: 2.9622 - val_loss: 2.2676


Epoch 2/8


9/9 - 4s - 469ms/step - loss: 2.1562 - val_loss: 1.9552


Epoch 3/8


...

9/9 - 3s - 312ms/step - loss: 1.4462 - val_loss: 1.3106


Epoch 7/8


9/9 - 2s - 266ms/step - loss: 1.3020 - val_loss: 1.1563


Epoch 8/8


9/9 - 3s - 280ms/step - loss: 1.1546 - val_loss: 1.0066


## 5. 입력에 따른 출력 생성 확인

학습된 모델에 시작 문자열(prompt)을 주고, **다음 토큰을 한 개씩 예측·이어붙이며** 텍스트를 생성한다.
이것이 causal LM의 동작 방식이다: 지금까지의 토큰만 보고 다음을 만든다.
(출력 결과물의 품질이 아니라 **모델이 정상적으로 다음 토큰을 생성하는지** 확인하는 것이 목적이다.)

In [9]:
def generate(prompt, n_new=120, temperature=0.8):
    ids = encode(prompt)
    for _ in range(n_new):
        # context를 SEQ_LEN으로 맞추기 (왼쪽 패딩)
        ctx = ids[-SEQ_LEN:]
        pad = [0] * (SEQ_LEN - len(ctx))
        inp = np.array([pad + ctx], dtype=np.int32)
        logits = model(inp, training=False)[0, -1].numpy()   # 마지막 위치의 다음토큰 분포
        logits = logits / temperature
        probs = tf.nn.softmax(logits).numpy()
        next_id = np.random.choice(len(probs), p=probs)       # 확률 샘플링
        ids.append(int(next_id))
    return decode(ids)

print("=== 생성 결과 ===")
print(generate("the ", n_new=150))

=== 생성 결과 ===


the mpamamarmo.eneamamoframokeamode. trokepamodeararmathernt. de. th lat th atheramath de psfrasfrneamacke ly ly de therndeathe fror lframokekeskeamacth l


---
### 정리

- **평가기준 1**: 위 표/서술로 원조 Transformer 대비 변경점(Encoder·cross-attn 제거, causal mask, 학습형 position embedding)을 명시.
- **평가기준 2**: 한 칸 shift로 (입력, 다음토큰 정답) 쌍 생성.
- **평가기준 3**: `TokenAndPositionEmbedding`으로 위치 정보 추가 (논문 식 h0 = We + Wp).
- **평가기준 4**: `GPTBlock`(cross-attn 없는 masked self-attn 블록) → `build_gpt` → `model.summary()` / `model.fit()`.
- **평가기준 5**: `generate()`로 입력에 따른 다음 토큰 생성 확인.